In [48]:
import os
import pandas as pd

In [49]:
dir_path = 'outputs'
# add class column for attack
paths = os.listdir(dir_path)
for path in paths:
    df = pd.read_csv(f'{dir_path}/{path}')
    df['class'] = path.split('_')[0]
    df.to_csv(f'crawl/raw/{path}', index=False)

In [50]:
# combine data in one
dir_path = 'crawl/raw'
paths = os.listdir(dir_path)
data = []
for path in paths:
    df = pd.read_csv(f'{dir_path}/{path}')
    df_selected = df[['frame.len', 'eth.src', 'eth.dst', 
                      'ip.proto', 'ip.flags', 'ip.ttl', 
                      'tcp.srcport', 'tcp.dstport', 'tcp.flags',
                      'class']]
    data.append(df_selected)
combined_df = pd.concat(data, axis=0)
combined_df.to_csv(f'{dir_path}/combined_data.csv', index=False)

In [37]:
# preprocess data
"""
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
categorical_columns = ['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags']

path = 'crawl/raw/combined_data.csv'
df = pd.read_csv(path)
print(df.shape)
print(df.info())

#unique_counts = df.nunique()
#unique_counts

# remove dupplicate
df.drop_duplicates(inplace=True)
print(df.shape)

# remove nan value
df.dropna(inplace=True)
print(df.shape)

# one-hot encoding
df = pd.get_dummies(df, columns=['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags'], drop_first=True)
df.head()

# df = df.drop()
# remove columns have one value
df = df.loc[:, df.nunique() > 1]

df['class'] = df['class'].map({'normal': 0, 'ddos': 1, 'dos': 2, 'scanport': 3})
df['class'].unique()
# df.to_csv(f'crawl/raw/data.csv', index=False)
#y = df['class']
#X = df.drop(['class', 'ip.src', 'ip.dst'], axis=1)
#print(X.shape, y.shape)
"""

"\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.compose import ColumnTransformer\ncategorical_columns = ['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags']\n\npath = 'crawl/raw/combined_data.csv'\ndf = pd.read_csv(path)\nprint(df.shape)\nprint(df.info())\n\n#unique_counts = df.nunique()\n#unique_counts\n\n# remove dupplicate\ndf.drop_duplicates(inplace=True)\nprint(df.shape)\n\n# remove nan value\ndf.dropna(inplace=True)\nprint(df.shape)\n\n# one-hot encoding\ndf = pd.get_dummies(df, columns=['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags'], drop_first=True)\ndf.head()\n\n# df = df.drop()\n# remove columns have one value\ndf = df.loc[:, df.nunique() > 1]\n\ndf['class'] = df['class'].map({'normal': 0, 'ddos': 1, 'dos': 2, 'scanport': 3})\ndf['class'].unique()\n# df.to_csv(f'crawl/raw/data.csv', index=False)\n#y = df['class']\n#X = df.drop(['class', 'ip.src', 'ip.dst'], axis=1)\n#print(X.shape, y.shape)\n"

In [99]:
# test
# preprocess data
import os
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle

#path = 'crawl/raw/combined_data.csv'
#df = pd.read_csv(path)
categorical_columns = ['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags']
numeric_columns = ['frame.len', 'ip.ttl', 'tcp.srcport', 'tcp.dstport']
df = pd.read_csv(f'crawl/raw/combined_data.csv')
df = df[['frame.len', 'eth.src', 'eth.dst', 
                      'ip.flags', 'ip.ttl', 
                      'tcp.srcport', 'tcp.dstport', 'tcp.flags',
                      'class']]
# remove columns have one value
df['class'] = df['class'].map({'normal': 0, 'ddos': 1, 'dos': 2, 'scanport': 3})
y = df['class']
df = df.drop(['class'], axis=1)

#df_selected = df[['frame.len', 'eth.src', 'eth.dst', 'ip.src', 'ip.dst',  'ip.proto', 'ip.flags', 'ip.ttl', 
#                'tcp.srcport', 'tcp.dstport', 'tcp.flags', 'class']]

# remove dupplicate
df.drop_duplicates(inplace=True)

# remove nan value
df.dropna(inplace=True)

# one-hot encoding
#df = pd.get_dummies(df, columns=['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags'], drop_first=True)
encoder = OneHotEncoder(handle_unknown='ignore', drop='first')
preprocessor = ColumnTransformer(
    transformers=[('onehot', encoder, categorical_columns)],
    remainder='passthrough'
)

# Fit và transform
df = preprocessor.fit_transform(df)

# Lấy tên cột mới
encoded_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(categorical_columns)
original_column_names = numeric_columns  # Các cột không bị biến đổi

# Kết hợp tên cột mới và cột gốc
all_column_names = list(encoded_feature_names) + original_column_names

# Tạo DataFrame với tên cột đúng
df = pd.DataFrame(df, columns=all_column_names)

import pickle

# Lưu preprocessor đã fit vào file pickle
with open('crawl/raw/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

In [100]:
import pandas as pd
import pickle
def preprocess_data(csv_path):
    df = pd.read_csv(csv_path)
    
    categorical_columns = ['eth.src', 'eth.dst', 'ip.flags', 'tcp.flags']
    numeric_columns = ['frame.len', 'ip.ttl', 'tcp.srcport', 'tcp.dstport']
    df = df[['frame.len', 'eth.src', 'eth.dst', 'ip.flags', 'ip.ttl', 'tcp.srcport', 'tcp.dstport', 'tcp.flags']]

    df.drop_duplicates(inplace=True)
    df.dropna(inplace=True)
    
    with open('crawl/raw/preprocessor.pkl', 'rb') as f:
        preprocessor = pickle.load(f)
    df = preprocessor.transform(df)
    encoded_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(categorical_columns)
    original_column_names = numeric_columns
    all_column_names = list(encoded_feature_names) + original_column_names
    df = pd.DataFrame(df, columns=all_column_names)
    return df

In [102]:
df = preprocess_data(csv_path='crawl/raw/dos_attack.csv')
df.shape

(19920, 15)

In [83]:
"""
df = pd.read_csv(f'crawl/raw/normal_network.csv')
df = df[['frame.len', 'eth.src', 'eth.dst', 
                      'ip.flags', 'ip.ttl', 
                      'tcp.srcport', 'tcp.dstport', 'tcp.flags',
                      'class']]
print(df.shape)
df = preprocessor.transform(df)
print(df.shape)
encoded_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(categorical_columns)
original_column_names = numeric_columns  # Các cột không bị biến đổi

# Kết hợp tên cột mới và cột gốc
all_column_names = list(encoded_feature_names) + original_column_names

# Tạo DataFrame với tên cột đúng
df = pd.DataFrame(df, columns=all_column_names)
print(df.shape)

df.shape
"""

(2166, 9)
(2166, 16)
(2166, 16)


C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


(2166, 16)

In [90]:
from utils.dataset_utils import check, separate_data, split_data, save_file
from sklearn.preprocessing import MinMaxScaler
import random 
import numpy as np

random.seed(42)
np.random.seed(42)
num_clients = 20
dir_path = "crawl/"
niid = False
balance = 'dir'
partition = 'exdir'

X = X.to_numpy()

X = np.array(X, dtype=np.float64)

# normalize data

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

y = y.to_numpy()
num_classes = len(set(y))
print("Original Size: ", (X.shape, y.shape))
print(f'Number of classes: {num_classes}')

X, y, statistic = separate_data((X, y), num_clients, num_classes, niid, balance, partition, class_per_client=2)

train_data, test_data, = split_data(X, y)
data_type = 'noniid'
config_path = dir_path + data_type + "/config.json"
train_path = dir_path + data_type + "/train/"
test_path = dir_path + data_type + "/test/"

save_file(config_path, train_path, test_path, train_data, test_data, num_clients, num_classes, statistic, balance)

AttributeError: 'list' object has no attribute 'to_numpy'

In [89]:
X[0].shape

(8524, 15)